# Taxol active spaces: a complete small proof of concept

This notebook starts with **9WDA, 9WDB and 1JFF**, builds a defined molecular model, calculates electronic active spaces, and exports files for qubit mapping. **No blank chemistry forms or manual force-field templates are needed for this supported recipe.**

The model is deliberately bounded: a 31-atom Taxol/His contact is quantum mechanical; the remaining Taxol and capped nearby peptide fragments provide fixed MMFF94 charges. The environment explicitly includes the position corresponding to the human S239C mutation. It is a **static finite embedded cluster**, not a full solvated protein simulation or a binding-affinity calculation.

**Start here:** read README.md, use this notebook's folder as the working directory, and run all cells in order. Existing verified calculations are reused only when inputs and output hashes match. Set `force_recalculate=True` to repeat the electronic calculations. Full recomputation is substantially slower than a cached run.

All current results are under **active_space_outputs/completed/**. Earlier output folders from the manual-input version are not used. The notebook contains all its own functions; there is no separate helper-script dependency.

Pocket = nearby atoms. QM region = chemically complete atoms. Active space = selected electrons and orbitals within the QM region.

## 1. Install once

The existing `.venv-taxol-notebook` environment has been prepared for this project. For a fresh machine, install the packages below in the notebook kernel's environment, then restart the kernel. ChimeraX is optional for viewing the exported structures; it is not required to run the workflow.

In [1]:
# Uncomment only when setting up a fresh environment, then restart the kernel.
# %pip install numpy scipy biopython gemmi rdkit pyscf qiskit qiskit-nature nbformat


## 2. Settings and small utilities

Keep the defaults for the delivered example. `base=0.10` and `expanded=0.08` are AVAS projection thresholds, not numbers of orbitals. The lower threshold retains more orbital character. No arbitrary orbital truncation is used to achieve a qubit target.

The HIE histidine state puts a hydrogen on NE2. This is an explicit chemical hypothesis, not a calculated pKa. Other titratable groups use the documented conventional states. Changing chemistry requires validating a new model, even though construction is automated.

In [2]:
from pathlib import Path
from collections import defaultdict
from itertools import product, combinations
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError
import json, csv, hashlib, math, time, re, platform, importlib.metadata as metadata
import numpy as np
import gemmi
from rdkit import Chem
from rdkit.Chem import AllChem
from pyscf import gto, scf, qmmm, mcscf, ao2mo, fci, lib
from pyscf.mcscf import avas
from pyscf.tools import molden, fcidump, cubegen

CFG = dict(ids=['9WDA','9WDB','1JFF'], output='active_space_outputs/completed',
           pocket_A=5.0, histidine='HIE', basis='def2-svp',
           thresholds={'base':0.1,'expanded':0.08},
           max_qm_atoms=60, max_aos=400, max_active=12,
           max_determinants=250000, threads=2, memory_MB=2000,
           force_recalculate=False, make_cubes=False)
OUT=Path(CFG['output']).resolve(); INPUT=OUT/'input'
INPUT.mkdir(parents=True,exist_ok=True)
STANDARD_AA=set('ALA ARG ASN ASP CYS GLN GLU GLY HIS ILE LEU LYS MET PHE PRO SER THR TRP TYR VAL'.split())
VERSIONS={p:metadata.version(p) for p in ['numpy','scipy','gemmi','rdkit','pyscf','qiskit','qiskit-nature']}
lib.num_threads(CFG['threads'])
def check(value, message):
    if not value: raise ValueError(message)
def save(path,data):
    Path(path).write_text(json.dumps(data,indent=2,default=lambda x:x.tolist() if hasattr(x,'tolist') else str(x))+'\n')
def sha(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def fingerprint(data):return hashlib.sha256(json.dumps(data,sort_keys=True,default=lambda x:x.tolist() if hasattr(x,'tolist') else str(x)).encode()).hexdigest()
def number(value, default=None):
    try:return float(value)
    except (TypeError,ValueError):return default

def rows(block, category):
    columns=block.get_mmcif_category(category)
    return [dict(zip(columns,values)) for values in zip(*columns.values())] if columns else []

def download(url,destination):
    path=Path(destination)
    if not path.exists():
        path.parent.mkdir(parents=True,exist_ok=True)
        with urlopen(Request(url,headers={'User-Agent':'TaxolActiveSpace/2.0'}),timeout=60) as r:data=r.read()
        check(data.startswith(b'data_'),'Download is not a chemical mmCIF file')
        path.write_bytes(data)
    return path


/Users/srijan/Documents/ChatGPT/Quantum Computing/.venv-taxol-notebook/lib/python3.14/site-packages/pyscf/lib/misc.py:203: UserWarning: OpenMP is not available. Setting omp_threads to 2 has no effects.
  warnings.warn('OpenMP is not available. '


## 3. Read the deposited assemblies and locate Taxol

Gemmi reads assembly 1 and its symmetry operators, preserving both author and label residue numbers. Alternate conformations are chosen coherently by occupancy. We identify bound TA1, search within 5 Å, and later use the union of corresponding protein positions across the three structures. Filling one-residue gaps prevents overlapping peptide caps.

Missing required heavy atoms cause an explicit error. For the delivered structures, the selected components pass the complete-atom checks; no heavy-atom repair is needed.

In [3]:
def operation_sequences(expression):
    """Expand deposited operation ranges and products, such as (1-3)(4,5)."""
    groups = re.findall(r"\(([^()]+)\)", expression) or [expression]
    expanded = []
    for group in groups:
        values = []
        for item in group.split(","):
            item = item.strip()
            if re.fullmatch(r"\d+-\d+", item):
                first, last = map(int, item.split("-"))
                values.extend(str(x) for x in range(first, last + 1))
            else:
                values.append(item)
        expanded.append(values)
    return list(product(*expanded))


def assembly_atoms(block, assembly_id="1", model_id="1", max_atoms=300000):
    """Apply CIF assembly operators and keep original atom identifiers."""
    source = []
    for row in rows(block, "_atom_site."):
        if str(row.get("pdbx_PDB_model_num", "1")) != str(model_id):
            continue
        if row.get("type_symbol", "").upper() in {"H", "D"}:
            continue  # Hydrogen placement is regenerated during preparation.
        seq = row.get("label_seq_id") or row.get("auth_seq_id")
        source.append({"chain": row["label_asym_id"], "auth_chain": row.get("auth_asym_id"),
                       "entity": row["label_entity_id"], "seq": str(seq),
                       "auth_seq": row.get("auth_seq_id"), "insertion": row.get("pdbx_PDB_ins_code"),
                       "resname": row["label_comp_id"], "name": row["label_atom_id"],
                       "element": row["type_symbol"], "alt": row.get("label_alt_id") or "",
                       "occupancy": number(row.get("occupancy"), 0),
                       "b_factor": number(row.get("B_iso_or_equiv")),
                       "xyz": np.array([float(row[k]) for k in ("Cartn_x", "Cartn_y", "Cartn_z")]),
                       "source_id": row["id"]})
    # Resolve each residue coherently; blank atoms coexist with one selected alt.
    grouped = defaultdict(list)
    for atom in source:
        grouped[(atom["chain"], atom["seq"], atom["insertion"], atom["resname"])].append(atom)
    selected = []
    for atoms in grouped.values():
        scores = defaultdict(float)
        for atom in atoms:
            if atom["alt"]:
                scores[atom["alt"]] += atom["occupancy"]
        choice = sorted(scores, key=lambda x: (-scores[x], x))[0] if scores else ""
        for atom in atoms:
            if atom["alt"] in {"", choice}:
                atom["alternate_present"] = bool(scores)
                selected.append(atom)
    operations = {}
    for row in rows(block, "_pdbx_struct_oper_list."):
        matrix = np.eye(4)
        for i in range(3):
            matrix[i, 3] = float(row[f"vector[{i+1}]"])
            for j in range(3):
                matrix[i, j] = float(row[f"matrix[{i+1}][{j+1}]"])
        operations[row["id"]] = matrix
    generators = [x for x in rows(block, "_pdbx_struct_assembly_gen.")
                  if str(x["assembly_id"]) == str(assembly_id)]
    if not generators:
        raise ValueError(f"Deposited assembly {assembly_id} is unavailable.")
    output, seen = [], set()
    for generator in generators:
        chains = set(generator["asym_id_list"].split(","))
        for sequence in operation_sequences(generator["oper_expression"]):
            transform = np.eye(4)
            for operation in sequence:
                transform = transform @ operations[operation]
            for atom in selected:
                if atom["chain"] not in chains:
                    continue
                xyz = (transform @ np.append(atom["xyz"], 1))[:3]
                key = (atom["source_id"], tuple(np.round(xyz, 5)))
                if key in seen:
                    continue
                seen.add(key)
                copy = dict(atom, xyz=xyz, operation="*".join(sequence))
                copy["residue_key"] = f'{copy["chain"]}:{copy["operation"]}:{copy["seq"]}:{copy["insertion"]}:{copy["resname"]}'
                output.append(copy)
                if len(output) > max_atoms:
                    raise ValueError("Assembly exceeds configured atom budget; choose a smaller deposited assembly.")
    return output


def extract_pockets(atoms, beta_entities, radius):
    """Enumerate bound TA1 sites and select only complete local residues."""
    residues = defaultdict(list)
    for atom in atoms:
        residues[atom["residue_key"]].append(atom)
    pockets = []
    for ligand_key, ligand in residues.items():
        if ligand[0]["resname"] != "TA1":
            continue
        ligand_xyz = np.array([a["xyz"] for a in ligand])
        contacts, selected, nearest_beta = [], [], []
        for key, residue in residues.items():
            if key == ligand_key:
                continue
            xyz = np.array([a["xyz"] for a in residue])
            distances = np.linalg.norm(xyz[:, None, :] - ligand_xyz[None, :, :], axis=2)
            i, j = np.unravel_index(distances.argmin(), distances.shape)
            distance = float(distances[i, j])
            if residue[0]["entity"] in beta_entities:
                nearest_beta.append((distance, residue[0]["chain"], residue[0]["operation"]))
            if distance <= radius:
                contacts.append({"residue_key": key, "resname": residue[0]["resname"],
                                 "protein_atom": residue[i]["name"], "ligand_atom": ligand[j]["name"],
                                 "distance_angstrom": distance,
                                 "within_4A": distance <= 4, "within_5A": distance <= 5})
                if residue[0]["resname"] in STANDARD_AA:
                    selected.append(key)
        if not nearest_beta or min(nearest_beta)[0] > radius:
            pockets.append({"ligand_key": ligand_key, "status": "rejected",
                            "reasons": ["TA1 is not near a verified beta-tubulin entity."]})
            continue
        pockets.append({"ligand_key": ligand_key, "selected_residues": selected,
                        "contacts": contacts, "beta_instance": min(nearest_beta)[1:],
                        "status": "extracted", "reasons": [], "ligand": ligand,
                        "residue_lookup": residues})
    # Include a single intervening residue to prevent overlapping cap atoms.
    for pocket in pockets:
        if pocket["status"] != "extracted":
            continue
        selected = set(pocket["selected_residues"])
        by_position = {(r[0]["chain"], r[0]["operation"], int(r[0]["seq"])): k
                       for k, r in residues.items() if r[0]["resname"] in STANDARD_AA}
        additions = []
        for (chain, operation, seq), key in by_position.items():
            if key not in selected and by_position.get((chain, operation, seq-1)) in selected and by_position.get((chain, operation, seq+1)) in selected:
                additions.append(key)
        pocket["bridging_residues"] = additions
        pocket["selected_residues"] += additions
    return pockets

## 4. Build real chemical bonds and choose protonation

Bonds and atom identities come from the PDB Chemical Component Dictionary, not guessed distances. Asp/Glu are negatively charged; Lys/Arg positively charged; His is HIE and Cys neutral in this recipe. Every choice is saved.

The code checks Taxol's stereochemistry against the chemical definition. This is structural model validation, not proof of the protonation state in a real protein.

In [4]:
def component_definition(component, cache_dir):
    """Use deposited chemical topology, not guessed distance-based bonds."""
    path = download(f"https://files.rcsb.org/ligands/download/{component}.cif",
                    Path(cache_dir) / "ccd" / f"{component}.cif")
    block = gemmi.cif.read_file(str(path)).sole_block()
    return rows(block, "_chem_comp_atom."), rows(block, "_chem_comp_bond.")


def add_component(builder, coordinates, component, cache_dir, mapping, positions):
    """Add all observed heavy atoms with CCD bonds and formal charges."""
    definitions, bonds = component_definition(component, cache_dir)
    expected = {a["atom_id"]: a for a in definitions
                if a["type_symbol"] not in {"H", "D"} and not (component in STANDARD_AA and a["atom_id"] == "OXT")}
    actual = {a["name"]: a for a in coordinates}
    if len(actual) != len(coordinates):
        raise ValueError(f"Duplicate atom names in {component}.")
    if set(actual) != set(expected):
        raise ValueError(f"{component}: missing {sorted(set(expected)-set(actual))}, unexpected {sorted(set(actual)-set(expected))}")
    indices = {}
    for name, definition in expected.items():
        atom = Chem.Atom(definition["type_symbol"])
        atom.SetFormalCharge(int(number(definition.get("charge"), 0)))
        # Backbone nitrogens become neutral peptide amides after cap/link assembly.
        if component in STANDARD_AA and name == "N":
            atom.SetFormalCharge(0)
        index = builder.AddAtom(atom)
        indices[name] = index
        record = actual[name]
        positions.append(record["xyz"])
        mapping.append({"kind": "experimental", "source_id": record["source_id"],
                        "residue_key": record["residue_key"], "atom_name": name,
                        "operation": record["operation"],
                        "expected_stereo": definition.get("pdbx_stereo_config") if component == "TA1" else None})
    orders = {"SING": Chem.BondType.SINGLE, "DOUB": Chem.BondType.DOUBLE,
              "TRIP": Chem.BondType.TRIPLE, "AROM": Chem.BondType.AROMATIC}
    for bond in bonds:
        left, right = bond["atom_id_1"], bond["atom_id_2"]
        if left in indices and right in indices:
            builder.AddBond(indices[left], indices[right], orders[bond["value_order"]])
    return indices


def apply_protonation(builder, indices, residue_name, choice=None):
    """Explicit conventional pH-7.4 microstates; HIS/CYS require a choice."""
    defaults = {"ASP": "deprotonated", "GLU": "deprotonated", "LYS": "protonated", "ARG": "protonated"}
    if residue_name in {"HIS", "CYS"} and choice is None:
        raise ValueError(f"{residue_name} protonation is ambiguous; supply a residue-key override.")
    choice = choice or defaults.get(residue_name, "neutral")
    if residue_name in {"ASP", "GLU"}:
        center, oxygen = ("CG", "OD2") if residue_name == "ASP" else ("CD", "OE2")
        first = "OD1" if residue_name == "ASP" else "OE1"
        if choice not in {"deprotonated", "neutral"}:
            raise ValueError("Acid choice must be deprotonated or neutral.")
        builder.GetBondBetweenAtoms(indices[center], indices[first]).SetBondType(Chem.BondType.DOUBLE)
        builder.GetBondBetweenAtoms(indices[center], indices[oxygen]).SetBondType(Chem.BondType.SINGLE)
        builder.GetAtomWithIdx(indices[first]).SetFormalCharge(0)
        builder.GetAtomWithIdx(indices[oxygen]).SetFormalCharge(-1 if choice == "deprotonated" else 0)
    elif residue_name == "LYS":
        if choice not in {"protonated", "neutral"}:
            raise ValueError("LYS choice must be protonated or neutral.")
        builder.GetAtomWithIdx(indices["NZ"]).SetFormalCharge(1 if choice == "protonated" else 0)
    elif residue_name == "ARG":
        if choice != "protonated":
            raise ValueError("Only protonated ARG is supported by this template.")
        # Guanidinium charge is assigned to its double-bonded terminal nitrogen.
        for name in ("NE", "NH1", "NH2"):
            builder.GetAtomWithIdx(indices[name]).SetFormalCharge(0)
        double_n = next(n for n in ("NH1", "NH2") if builder.GetBondBetweenAtoms(indices["CZ"], indices[n]).GetBondTypeAsDouble() == 2)
        builder.GetAtomWithIdx(indices[double_n]).SetFormalCharge(1)
    elif residue_name == "HIS":
        if choice not in {"HID", "HIE", "HIP"}:
            raise ValueError("HIS choices: HID (ND1-H), HIE (NE2-H), HIP (both).")
        ring = [("CG", "ND1"), ("ND1", "CE1"), ("CE1", "NE2"), ("NE2", "CD2"), ("CD2", "CG")]
        for left, right in ring:
            bond = builder.GetBondBetweenAtoms(indices[left], indices[right])
            bond.SetBondType(Chem.BondType.AROMATIC)
            bond.SetIsAromatic(True)
        for name in ("CG", "ND1", "CE1", "NE2", "CD2"):
            builder.GetAtomWithIdx(indices[name]).SetIsAromatic(True)
        for name in ("ND1", "NE2"):
            atom = builder.GetAtomWithIdx(indices[name])
            has_h = choice == "HIP" or (name == "ND1" and choice == "HID") or (name == "NE2" and choice == "HIE")
            atom.SetFormalCharge(1 if choice == "HIP" and name == "NE2" else 0)
            atom.SetNumExplicitHs(int(has_h))
            atom.SetNoImplicit(True)
    elif residue_name == "CYS":
        if choice not in {"neutral", "deprotonated"}:
            raise ValueError("CYS choice must be neutral or deprotonated.")
        builder.GetAtomWithIdx(indices["SG"]).SetFormalCharge(-1 if choice == "deprotonated" else 0)
    elif choice != "neutral":
        raise ValueError(f"Unsupported microstate for {residue_name}: {choice}")
    return choice

## 5. Cap peptide segments and relax hydrogens

The classical environment contains complete Taxol plus selected peptide segments. ACE and NME caps complete exposed peptide ends using neighbouring backbone positions. RDKit checks valence and MMFF94 parameter coverage, adds hydrogens and relaxes **only hydrogens**. Experimental heavy atoms and heavy cap anchors remain fixed.

MMFF94 is used for this finite organic fragment model. It replaces the unfinished requirement to supply custom OpenMM templates. It is not a claim that MMFF94 describes a full protein better than a biomolecular force field.

In [5]:
def prepare_region(pocket, cache_dir, protonation_overrides, max_atoms=1800):
    """Prepare full TA1 plus selected peptide segments, with ACE/NME end caps."""
    lookup, selected = pocket["residue_lookup"], pocket["selected_residues"]
    builder, positions, mapping, indices = Chem.RWMol(), [], [], {}
    caps, microstates = [], {}
    add_component(builder, pocket["ligand"], "TA1", cache_dir, mapping, positions)
    for key in selected:
        atoms = lookup[key]
        indices[key] = add_component(builder, atoms, atoms[0]["resname"], cache_dir, mapping, positions)
        microstates[key] = apply_protonation(builder, indices[key], atoms[0]["resname"], protonation_overrides.get(key))
    # Original backbone atoms determine peptide neighbors without renumbering.
    residue_by_seq = {(r[0]["chain"], r[0]["operation"], r[0]["seq"]): key
                      for key, r in lookup.items() if r[0]["resname"] in STANDARD_AA}
    def neighbor(key, offset):
        atom = lookup[key][0]
        seq = int(atom["seq"]) + offset
        return residue_by_seq.get((atom["chain"], atom["operation"], str(seq)))
    def atom_record(key, name):
        if key is None:
            raise ValueError("Missing backbone neighbor: cannot construct a geometry-based cap.")
        candidates = [a for a in lookup[key] if a["name"] == name]
        if len(candidates) != 1:
            raise ValueError(f"Missing/ambiguous cap anchor {key}:{name}.")
        return candidates[0]
    def link(left_key, right_key):
        a, b = atom_record(left_key, "C"), atom_record(right_key, "N")
        if not 1.1 < np.linalg.norm(a["xyz"] - b["xyz"]) < 1.7:
            raise ValueError("Broken peptide geometry at a truncation boundary.")
    def cap_atom(key, name, element, cap_kind):
        record = atom_record(key, name)
        index = builder.AddAtom(Chem.Atom(element))
        positions.append(record["xyz"])
        mapping.append({"kind": "cap", "cap": cap_kind, "source_anchor_id": record["source_id"],
                        "residue_key": key, "atom_name": name})
        return index
    for key in selected:
        previous, following = neighbor(key, -1), neighbor(key, 1)
        link(previous, key)
        link(key, following)
        if previous in indices:
            builder.AddBond(indices[previous]["C"], indices[key]["N"], Chem.BondType.SINGLE)
        else:
            # ACE replaces the excluded previous residue: methyl-C(=O)-N.
            methyl = cap_atom(previous, "CA", "C", "ACE")
            carbon = cap_atom(previous, "C", "C", "ACE")
            oxygen = cap_atom(previous, "O", "O", "ACE")
            builder.AddBond(methyl, carbon, Chem.BondType.SINGLE)
            builder.AddBond(carbon, oxygen, Chem.BondType.DOUBLE)
            builder.AddBond(carbon, indices[key]["N"], Chem.BondType.SINGLE)
            caps.append({"type": "ACE", "attached_to": key})
        if following not in indices:
            # NME replaces the excluded next residue: C(=O)-NH-methyl.
            nitrogen = cap_atom(following, "N", "N", "NME")
            methyl = cap_atom(following, "CA", "C", "NME")
            builder.AddBond(indices[key]["C"], nitrogen, Chem.BondType.SINGLE)
            builder.AddBond(nitrogen, methyl, Chem.BondType.SINGLE)
            caps.append({"type": "NME", "attached_to": key})
    molecule = builder.GetMol()
    Chem.SanitizeMol(molecule)
    conformer = Chem.Conformer(len(positions))
    for i, xyz in enumerate(positions):
        conformer.SetAtomPosition(i, tuple(map(float, xyz)))
    molecule.AddConformer(conformer)
    Chem.AssignStereochemistryFrom3D(molecule)
    Chem.AssignStereochemistry(molecule, cleanIt=True, force=True)
    for index, provenance in enumerate(mapping):
        expected = provenance.get("expected_stereo")
        if expected in {"R", "S"}:
            atom = molecule.GetAtomWithIdx(index)
            if not atom.HasProp("_CIPCode") or atom.GetProp("_CIPCode") != expected:
                raise ValueError("Taxol stereochemistry disagrees with the CCD at " + provenance["atom_name"])
    # Preserve bound heavy atoms, add hydrogens, and relax only the new hydrogens.
    heavy_count = molecule.GetNumAtoms()
    molecule = Chem.AddHs(molecule, addCoords=True)
    if molecule.GetNumAtoms() > max_atoms:
        raise ValueError("Prepared region exceeds configured atom budget.")
    if not AllChem.MMFFHasAllMoleculeParams(molecule):
        raise ValueError("MMFF94 lacks parameters for the prepared cluster.")
    properties = AllChem.MMFFGetMoleculeProperties(molecule, mmffVariant='MMFF94')
    force_field = AllChem.MMFFGetMoleculeForceField(molecule, properties, ignoreInterfragInteractions=False)
    for i in range(heavy_count):
        force_field.AddFixedPoint(i)
    force_field.Initialize()
    converged = force_field.Minimize(maxIts=10000, forceTol=1e-4) == 0
    for i in range(heavy_count, molecule.GetNumAtoms()):
        parent = molecule.GetAtomWithIdx(i).GetNeighbors()[0].GetIdx()
        mapping.append({"kind": "added_hydrogen", "parent_index": parent})
    charge = sum(a.GetFormalCharge() for a in molecule.GetAtoms())
    electrons = sum(a.GetAtomicNum() for a in molecule.GetAtoms()) - charge
    radicals = sum(a.GetNumRadicalElectrons() for a in molecule.GetAtoms())
    if radicals or electrons % 2:
        raise ValueError("This closed-shell preparation produced radicals or an odd electron count.")
    xyz = molecule.GetConformer().GetPositions()
    if not np.isfinite(xyz).all():
        raise ValueError("Invalid coordinates after hydrogen preparation.")
    # A severe nonbonded clash blocks readiness; bonded/1-3 pairs are excluded.
    topological = Chem.GetDistanceMatrix(molecule)
    distances = np.linalg.norm(xyz[:, None] - xyz[None, :], axis=2)
    bad = np.argwhere(np.triu((distances < 1.0) & (topological > 2), 1))
    warnings = []
    periodic_table = Chem.GetPeriodicTable()
    for bond in molecule.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        ideal = periodic_table.GetRcovalent(molecule.GetAtomWithIdx(i).GetAtomicNum()) + periodic_table.GetRcovalent(molecule.GetAtomWithIdx(j).GetAtomicNum())
        if not 0.65 * ideal < distances[i, j] < 1.35 * ideal:
            warnings.append(f"Unusual bond length for atoms {i}-{j}.")
    if not converged:
        warnings.append("Hydrogen relaxation did not converge.")
    if len(bad):
        warnings.append(f"{len(bad)} severe nonbonded atom clashes (<1 Å).")
    elements = [a.GetSymbol() for a in molecule.GetAtoms()]
    region = {"elements": elements, "coordinates_angstrom": xyz.tolist(),
              "bonds": [(b.GetBeginAtomIdx(), b.GetEndAtomIdx(), b.GetBondTypeAsDouble()) for b in molecule.GetBonds()],
              "formal_charges": [a.GetFormalCharge() for a in molecule.GetAtoms()],
              "net_charge": charge, "spin_2s": 0, "electron_count": electrons,
              "source_atom_mapping": mapping, "caps": caps, "protonation": microstates,
              "environment": "isolated_cluster", "ready_for_integrals": not warnings,
              "chemistry_checks_passed": not warnings,
              "warnings": warnings, "orbital_indices": None, "active_electrons": None,
              "molecular_region": "complete_taxol_plus_capped_local_peptide_segments"}
    return molecule, region


def save_pocket_files(atoms, contacts, folder):
    """Export inspectable coordinates and a compact contact table."""
    document = gemmi.cif.Document()
    block = document.add_new_block("selected_pocket")
    tags = ["group_PDB", "id", "type_symbol", "label_atom_id", "label_alt_id",
            "label_comp_id", "label_asym_id", "label_entity_id", "label_seq_id",
            "Cartn_x", "Cartn_y", "Cartn_z", "occupancy", "B_iso_or_equiv",
            "auth_asym_id", "auth_seq_id", "pdbx_PDB_model_num"]
    loop = block.init_loop("_atom_site.", tags)
    for i, atom in enumerate(atoms, 1):
        chain = atom["chain"] + "_" + atom["operation"]
        values = ["ATOM" if atom["resname"] in STANDARD_AA else "HETATM", str(i),
                  atom["element"], atom["name"], ".", atom["resname"], chain,
                  atom["entity"], atom["seq"] if atom["resname"] in STANDARD_AA else ".",
                  *[f"{x:.6f}" for x in atom["xyz"]], str(atom["occupancy"]),
                  str(atom["b_factor"] or 0), chain, str(atom["auth_seq"]), "1"]
        loop.add_row([value if value == "." else gemmi.cif.quote(value) for value in values])
    document.write_file(str(folder / "pocket.cif"))
    with (folder / "contacts.csv").open("w", newline="") as stream:
        fields = ["residue_key", "resname", "protein_atom", "ligand_atom", "distance_angstrom", "within_4A", "within_5A"]
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(contacts)

## 6. Make the small quantum contact model

Keep Taxol's N-methylbenzamide group and a methylimidazole representation of His. Preserve the aromatic rings and amide; replace three peripheral single C-C cuts with link hydrogens. The resulting model has **31 atoms, neutral charge and 116 electrons** before active-space reduction.

The contact is Taxol O14 to His NE2. Human His227 corresponds to **author His229 in 1JFF**. The source mapping records the distinction. A short contact distance alone does not prove a hydrogen bond: the donor-H-acceptor angle is calculated and unfavourable geometry is flagged.

In [6]:
def contact_model(pocket, region):
    """N-methylbenzamide + methylimidazole; retain rings/amide, cap C-C cuts."""
    if not region.get('chemistry_checks_passed'):
        raise ValueError('Parent region failed chemistry checks.')
    mapping = region['source_atom_mapping']
    ligand = pocket['ligand_key']
    histidines = [k for k in pocket['selected_residues'] if k.split(':')[2] == '227'
                  and k.endswith(':HIS')]
    if len(histidines) != 1:
        raise ValueError('This explicit model requires a unique His227.')
    histidine = histidines[0]
    if region['protonation'].get(histidine) != 'HIE':
        raise ValueError('The NE2-H donor model requires the explicit HIE scenario.')
    names = {ligand: {'C29', 'N01', 'C30', 'O14', 'C31', 'C32', 'C33', 'C34', 'C35', 'C36'},
             histidine: {'CB', 'CG', 'ND1', 'CD2', 'CE1', 'NE2'}}
    keep = [i for i, a in enumerate(mapping) if a.get('kind') == 'experimental'
            and a.get('atom_name') in names.get(a.get('residue_key'), set())]
    if len(keep) != sum(map(len, names.values())):
        raise ValueError('Missing or duplicate contact-model atoms.')
    # Keep parent hydrogens (and their microstate) on the retained heavy atoms.
    keep += [i for i, a in enumerate(mapping)
             if a.get('kind') == 'added_hydrogen' and a['parent_index'] in keep]
    old_to_new = {old: new for new, old in enumerate(keep)}
    builder = Chem.RWMol()
    xyz = np.asarray(region['coordinates_angstrom'])
    positions = [xyz[i].copy() for i in keep]
    provenance = [dict(mapping[i], parent_region_index=i) for i in keep]
    for i in keep:
        atom = Chem.Atom(region['elements'][i])
        atom.SetFormalCharge(region['formal_charges'][i])
        atom.SetNoImplicit(True)
        builder.AddAtom(atom)
    orders = {1: Chem.BondType.SINGLE, 1.5: Chem.BondType.AROMATIC,
              2: Chem.BondType.DOUBLE, 3: Chem.BondType.TRIPLE}
    cuts = []
    for i, j, order in region['bonds']:
        inside_i, inside_j = i in old_to_new, j in old_to_new
        if inside_i and inside_j:
            builder.AddBond(old_to_new[i], old_to_new[j], orders[order])
        elif inside_i != inside_j:
            inside, outside = (i, j) if inside_i else (j, i)
            if order != 1 or region['elements'][inside] != 'C' or region['elements'][outside] != 'C':
                raise ValueError('Only peripheral single C-C bonds may be capped.')
            h = builder.AddAtom(Chem.Atom('H'))
            builder.AddBond(old_to_new[inside], h, Chem.BondType.SINGLE)
            direction = xyz[outside] - xyz[inside]
            positions.append(xyz[inside] + 1.09 * direction / np.linalg.norm(direction))
            cuts.append({'retained_parent_index': inside, 'removed_parent_index': outside})
            provenance.append({'kind': 'link_hydrogen', **cuts[-1]})
    mol = builder.GetMol()
    Chem.SanitizeMol(mol)
    if len(Chem.GetMolFrags(mol)) != 2 or any(a.GetNumRadicalElectrons() for a in mol.GetAtoms()):
        raise ValueError('Expected exactly two closed-shell molecular fragments.')
    conf = Chem.Conformer(len(positions))
    for i, position in enumerate(positions):
        conf.SetAtomPosition(i, tuple(map(float, position)))
    mol.AddConformer(conf)
    if not AllChem.MMFFHasAllMoleculeParams(mol):
        raise ValueError('Missing hydrogen-relaxation parameters.')
    properties = AllChem.MMFFGetMoleculeProperties(mol, mmffVariant='MMFF94')
    ff = AllChem.MMFFGetMoleculeForceField(mol, properties, ignoreInterfragInteractions=False)
    for a in mol.GetAtoms():
        if a.GetAtomicNum() != 1:
            ff.AddFixedPoint(a.GetIdx())
    ff.Initialize()
    if ff.Minimize(maxIts=10000, forceTol=1e-4) != 0:
        raise ValueError('Contact-model hydrogen relaxation did not converge.')
    coords = mol.GetConformer().GetPositions()
    topology = Chem.GetDistanceMatrix(mol)
    distances = np.linalg.norm(coords[:, None] - coords[None, :], axis=2)
    if not np.isfinite(coords).all() or np.any(np.triu((distances < 1.0) & (topology > 2), 1)):
        raise ValueError('Invalid coordinates or severe nonbonded clash.')
    targets = [next(old_to_new[i] for i in keep if mapping[i].get('residue_key') == key
                    and mapping[i].get('atom_name') == name)
               for key, name in [(ligand, 'O14'), (histidine, 'NE2')]]
    if not 2.4 < distances[tuple(targets)] < 3.5:
        raise ValueError('The specified O14-NE2 contact is outside the model range.')
    acceptor, donor = targets
    donor_h = [a.GetIdx() for a in mol.GetAtomWithIdx(donor).GetNeighbors() if a.GetAtomicNum() == 1]
    if len(donor_h) != 1:
        raise ValueError('NE2 must have exactly one donor hydrogen.')
    h = donor_h[0]
    v, w = coords[donor] - coords[h], coords[acceptor] - coords[h]
    angle = float(np.degrees(np.arccos(np.clip(np.dot(v,w)/np.linalg.norm(v)/np.linalg.norm(w), -1, 1))))
    warnings = list(region.get('warnings', []))
    if angle < 120:
        warnings.append('O14...H-NE2 angle <120 degrees; do not label this a validated hydrogen bond.')
    model = {'pdb_id': pocket['pdb_id'], 'pocket_id': pocket['pocket_id'],
             'parent_region_id': region['region_id'], 'parent_atom_count': len(mapping),
             'elements': [a.GetSymbol() for a in mol.GetAtoms()],
             'coordinates_angstrom': coords.tolist(), 'net_charge': Chem.GetFormalCharge(mol),
             'spin_2s': 0, 'source_atom_mapping': provenance, 'cuts': cuts,
             'target_atom_indices': targets, 'contact_distance_angstrom': float(distances[tuple(targets)]),
             'donor_h_acceptor_angle_degrees': angle, 'protonation': {histidine: 'HIE'},
             'warnings': warnings, 'environment': 'isolated_contact_fragments',
             'model': 'N-methylbenzamide + methylimidazole, bound heavy atoms fixed',
             'limitations': 'Local electronic demonstration; no protein embedding, solvent, binding energy or affinity.'}
    return mol, model



## 7. Assign environmental charges and check correspondence

MMFF94 supplies a partial charge for every prepared parent atom. Atoms retained in the QM region are removed from the point-charge environment. Charges on the outside cut endpoints and their hydrogens are zeroed. Small corrections are distributed over a documented buffer of up to six eligible MM heavy atoms in the same component, preserving the intended component charges.

The buffer uses the same geometric rule across structures; recipient identities and all changes are recorded. This boundary treatment is an approximation that needs sensitivity testing before quantitative interpretation. No nonzero point charge may be closer than 1.2 Å to a QM atom.

The two human environments differ at **Ser239/Cys239**. We explicitly include that site even though it is outside 5 Å. The corresponding 1JFF residue is label-position 239, author **Cys241**. Sequence alignment checks these correspondences before calculations.

In [7]:
def inspect_inputs():
    pockets={}; entries={}
    for pid in CFG['ids']:
        path=download(f'https://files.rcsb.org/download/{pid}.cif',INPUT/f'{pid}.cif')
        block=gemmi.cif.read_file(str(path)).sole_block()
        beta=[r['id'] for r in rows(block,'_entity.')
              if 'tubulin' in r.get('pdbx_description','').lower() and 'beta' in r.get('pdbx_description','').lower()]
        check(len(beta)==1,f'{pid}: expected one beta tubulin entity')
        atoms=assembly_atoms(block)
        sites=extract_pockets(atoms,beta,CFG['pocket_A'])
        check(len(sites)==1 and sites[0]['status']=='extracted',f'{pid}: ambiguous Taxol site')
        p=sites[0];p.update(pdb_id=pid,pocket_id=f'{pid}_assembly1_TA1')
        pockets[pid]=p
        entries[pid]={'structure_url':f'https://files.rcsb.org/download/{pid}.cif','sha256':sha(path),
                      'assembly':'1','contacts':p['contacts'],
                      'missing_coordinate_records':rows(block,'_pdbx_unobs_or_zero_occ_atoms.')+rows(block,'_pdbx_unobs_or_zero_occ_residues.'),
                      'alternate_atoms':sum(a.get('alternate_present',False) for a in atoms)}
    # Use corresponding deposited beta sequence positions in these aligned entries.
    common={239}
    for p in pockets.values():
        beta_chain,beta_op=p['beta_instance']
        check(all(p['residue_lookup'][k][0]['chain']==beta_chain for k in p['selected_residues']),
              'This recipe expects a beta-only local pocket')
        common.update(int(p['residue_lookup'][k][0]['seq']) for k in p['selected_residues'])
    # Fill single-residue gaps to keep ACE/NME caps from occupying the same space.
    common |= {i+1 for i in common if i+2 in common}
    for pid,p in pockets.items():
        chain,op=p['beta_instance']
        byseq={int(a[0]['seq']):k for k,a in p['residue_lookup'].items()
               if a[0]['chain']==chain and a[0]['operation']==op and a[0]['resname'] in STANDARD_AA}
        check(common <= set(byseq),f'{pid}: a required local residue is unmodelled')
        p['selected_residues']=[byseq[i] for i in sorted(common)]
        p['forced_residue239']=byseq[239]
        expected={'9WDA':'SER','9WDB':'CYS','1JFF':'CYS'}[pid]
        check(p['residue_lookup'][byseq[239]][0]['resname']==expected,'Residue239 identity mismatch')
        d=OUT/pid;d.mkdir(exist_ok=True)
        save(d/'structure.json',{**entries[pid],'common_beta_positions':sorted(common),
                                'environment_residues':p['selected_residues'],
                                'residue239':byseq[239],
                                'excluded_nearby_nonprotein':[c for c in p['contacts'] if c['resname'] not in STANDARD_AA]})
        print(pid,'environment residues:',len(common),'mutation/reference:',expected,flush=True)
    # Sequence alignment checks correspondence; author numbering can differ.
    from Bio.Align import PairwiseAligner
    from Bio.SeqUtils import seq1
    aligner=PairwiseAligner(mode='global',match_score=2,mismatch_score=-1,open_gap_score=-5,extend_gap_score=-0.5)
    def seq_records(p):
        chain,op=p['beta_instance']
        return sorted([(int(a[0]['seq']),a[0]['resname'],a[0]['auth_seq'])
                       for a in p['residue_lookup'].values()
                       if a[0]['chain']==chain and a[0]['operation']==op and a[0]['resname'] in STANDARD_AA])
    reference=seq_records(pockets['9WDA'])
    for pid,p in pockets.items():
        target=seq_records(p)
        aln=aligner.align(''.join(seq1(r[1]) for r in reference),''.join(seq1(r[1]) for r in target))[0]
        pairs=[(reference[i],target[j]) for i,j in zip(*aln.indices) if i>=0 and j>=0]
        correspondence={a[0]:b[0] for a,b in pairs}
        check(all(correspondence.get(i)==i for i in common), 'This fixed recipe needs a revised alignment mapping')
        save(OUT/pid/'alignment.json',{'reference':'9WDA','pairs':pairs,
             'fields':['label_sequence_position','residue_name','author_residue_number']})
        selected_atoms=p['ligand']+[a for k in p['selected_residues'] for a in p['residue_lookup'][k]]
        save_pocket_files(selected_atoms,p['contacts'],OUT/pid)
    # Check the matched human sequence, not just residue numbers.
    for pos in sorted(common):
        identity=[]
        for pid in ['9WDA','9WDB']:
            p=pockets[pid];key=next(k for k in p['selected_residues'] if int(p['residue_lookup'][k][0]['seq'])==pos)
            identity.append(p['residue_lookup'][key][0]['resname'])
        check(identity[0]==identity[1] or (pos==239 and identity==['SER','CYS']),
              f'Unexpected human sequence difference at {pos}: {identity}')
    return pockets,entries


def write_molecule(mol, path):
    with Chem.SDWriter(str(path.with_suffix('.sdf'))) as writer:writer.write(mol)
    Chem.MolToXYZFile(mol,str(path.with_suffix('.xyz')))


def make_embedding(parent,region,qm,model):
    check(AllChem.MMFFHasAllMoleculeParams(parent),'MMFF94 parameter coverage incomplete')
    props=AllChem.MMFFGetMoleculeProperties(parent,mmffVariant='MMFF94')
    original=np.array([props.GetMMFFPartialCharge(i) for i in range(parent.GetNumAtoms())])
    xyz=parent.GetConformer().GetPositions()
    check(abs(original.sum()-region['net_charge'])<1e-7,'MMFF charges do not sum to formal charge')
    retained={r['parent_region_index'] for r in model['source_atom_mapping'] if 'parent_region_index' in r}
    qmxyz=qm.GetConformer().GetPositions()
    charge=original.copy();charge[list(retained)]=0
    boundary=set()
    for cut in model['cuts']:
        outside=cut['removed_parent_index'];boundary.add(outside)
        boundary.update(a.GetIdx() for a in parent.GetAtomWithIdx(outside).GetNeighbors() if a.GetAtomicNum()==1)
    charge[list(boundary)]=0
    adjustments=[]
    for fragment in Chem.GetMolFrags(parent):
        remain=sorted(set(fragment)-retained)
        if not remain:continue
        # Each remaining capped component keeps its intended integer formal charge.
        target=sum(parent.GetAtomWithIdx(i).GetFormalCharge() for i in remain)
        delta=target-charge[remain].sum()
        if abs(delta)>1e-10:
            candidates=[i for i in remain if i not in boundary and parent.GetAtomWithIdx(i).GetAtomicNum()>1
                        and np.linalg.norm(xyz[i]-qmxyz,axis=1).min()>2.5]
            check(bool(candidates),'No safe MM atoms for boundary charge redistribution')
            # A deterministic local buffer: nearest six eligible heavy atoms to cuts.
            cutxyz=xyz[list(boundary & set(fragment))]
            if len(cutxyz):candidates.sort(key=lambda i:(np.linalg.norm(xyz[i]-cutxyz,axis=1).min(),i))
            selected=candidates[:6]
            charge[selected]+=delta/len(selected)
            adjustments.append({'component_atoms':len(fragment),'target_charge_e':target,
                                'total_correction_e':float(delta),'recipient_indices':selected,
                                'per_atom_correction_e':float(delta/len(selected))})
    mmindices=sorted(set(range(parent.GetNumAtoms()))-retained)
    nonzero=[i for i in mmindices if abs(charge[i])>1e-12]
    mindistance=float(np.linalg.norm(xyz[nonzero,None,:]-qmxyz[None,:,:],axis=2).min())
    check(mindistance>1.2,f'MM point charge too close to QM atoms: {mindistance}')
    check(abs(charge.sum()+model['net_charge']-region['net_charge'])<1e-7,'Combined formal charge changed')
    model['environment']='finite_MMFF94_point_charge_cluster'
    model['limitations']='Static capped local model; no solvent, sampling, binding free energy or complete allostery.'
    model['embedding']={'forcefield':'MMFF94','parent_formal_charge':region['net_charge'],
        'QM_charge':model['net_charge'],'MM_charge':float(charge.sum()),
        'excluded_QM_parent_indices':sorted(retained),'zeroed_boundary_indices':sorted(boundary),
        'redistribution':adjustments,'minimum_MM_QM_distance_A':mindistance,
        'atom_indices':mmindices,'coordinates_A':xyz[mmindices].tolist(),
        'charges_e':charge[mmindices].tolist(),'original_charges_e':original[mmindices].tolist()}
    check(len(model['elements'])==31 and model['net_charge']==0,'Unexpected contact composition')
    return model


def build_models(pockets,entries):
    models={}
    for pid,p in pockets.items():
        print('Preparing',pid,flush=True)
        microstates={k:CFG['histidine'] if a[0]['resname']=='HIS' else 'neutral'
                     for k,a in p['residue_lookup'].items() if a[0]['resname'] in {'HIS','CYS'}}
        parent,region=prepare_region(p,INPUT,microstates)
        check(region['chemistry_checks_passed'],f'{pid}: preparation failed: {region["warnings"]}')
        region['region_id']=f'{pid}_common_environment_HIE'
        qm,model=contact_model(p,region)
        model=make_embedding(parent,region,qm,model)
        model.update(source=entries[pid],recipe='benzamide/His227-HIE + common capped local environment + residue239',
                     parent_protonation=region['protonation'],parent_caps=region['caps'],
                     qm_smiles=Chem.MolToSmiles(Chem.RemoveHs(qm)),
                     preparation_forcefield='MMFF94; only hydrogen positions relaxed; heavy atoms fixed')
        d=OUT/pid
        write_molecule(parent,d/'environment');write_molecule(qm,d/'qm')
        save(d/'model.json',model)
        save(d/'parent_model.json',region)
        models[pid]=model
        print(pid,'QM atoms',len(model['elements']),'MM atoms',len(model['embedding']['charges_e']),
              'contact',round(model['contact_distance_angstrom'],3),'angle',round(model['donor_h_acceptor_angle_degrees'],1),flush=True)
    save(OUT/'preparation_summary.json',{p:{'qm_atoms':len(m['elements']),
         'mm_atoms':len(m['embedding']['charges_e']),'warnings':m['warnings']} for p,m in models.items()})
    return models


In [8]:
POCKETS, ENTRIES = inspect_inputs()
MODELS = build_models(POCKETS, ENTRIES)


9WDA environment residues: 29 mutation/reference: SER


9WDB environment residues: 29 mutation/reference: CYS


1JFF environment residues: 29 mutation/reference: CYS


Preparing 9WDA


9WDA QM atoms 31 MM atoms 720 contact 2.773 angle 125.5


Preparing 9WDB


9WDB QM atoms 31 MM atoms 720 contact 2.795 angle 132.5


Preparing 1JFF


1JFF QM atoms 31 MM atoms 721 contact 3.213 angle 81.7


## 8. Calculate orbitals and select active spaces

PySCF runs density-fitted Hartree-Fock with the environmental charges included, checks convergence and internal stability, then uses AVAS on the **O14 and NE2 2p functions**. The selected orbital count is a result of the calculation.

Checks include orbital orthonormality, active electron count, and nonnegative Lowdin weights on target atoms and link caps. Cap-dominated orbitals are rejected. Molden files and orbital-character CSVs are saved for visual follow-up; automatic checks do not substitute for expert visual interpretation.

CASCI uses exact integrals in those orbitals, including the environmental field and inactive electrons. Both base and expanded spaces are solved classically. The code checks reconstruction of the reference determinant, variational energy and correlated electron count. The default resource limits stop unsupported sizes rather than silently reducing them.

In [9]:
def run_scf(model, folder):
    xyz=np.array(model['coordinates_angstrom'])
    # Recenter all coordinates together to reduce large-coordinate roundoff.
    origin=xyz.mean(axis=0);xyz=xyz-origin
    emb=model['embedding']; mmxyz=np.array(emb['coordinates_A'])-origin
    charges=np.array(emb['charges_e']); nonzero=np.abs(charges)>1e-12
    check(len(xyz)<=CFG['max_qm_atoms'],'QM atom budget exceeded')
    mol=gto.M(atom=list(zip(model['elements'],xyz)),unit='Angstrom',basis=CFG['basis'],
              charge=model['net_charge'],spin=model['spin_2s'],verbose=4,
              output=str(folder/'scf.log'),max_memory=CFG['memory_MB'])
    check(mol.spin==0 and mol.nelectron%2==0,'Only closed-shell models are implemented')
    check(mol.nao_nr()<=CFG['max_aos'],'AO budget exceeded before SCF')
    mf=qmmm.mm_charge(scf.RHF(mol).density_fit(),mmxyz[nonzero],charges[nonzero],unit='Angstrom')
    mf.conv_tol=1e-10;mf.max_cycle=100;mf.chkfile=str(folder/'scf.chk')
    mf.kernel()
    if not mf.converged:mf=mf.newton().run()
    check(mf.converged,'SCF did not converge')
    _,_,stable,_=mf.stability(return_status=True)
    check(bool(stable),'RHF is internally unstable; no Hamiltonian exported')
    # Use exact Coulomb/exchange and ERIs in the density-fitted reference orbitals.
    exact=qmmm.mm_charge(scf.RHF(mol),mmxyz[nonzero],charges[nonzero],unit='Angstrom')
    exact.mo_coeff,exact.mo_occ,exact.mo_energy=mf.mo_coeff,mf.mo_occ,mf.mo_energy
    save(folder/'mean_field.json',{'density_fitted_RHF_Eh':mf.e_tot,'converged':True,'internally_stable':bool(stable),
        'atoms':mol.natm,'atomic_orbitals':mol.nao_nr(),'total_electrons':mol.nelectron,
        'origin_shift_A':origin,'exact_energy_on_DF_density_Eh':exact.energy_tot(dm=mf.make_rdm1())})
    return mf,exact


def select_and_solve(mf,exact,model,folder,label,threshold):
    labels=[rf'^{i}\s+{re.escape(model["elements"][i])}\s+2p' for i in model['target_atom_indices']]
    for p in labels:check(len(mf.mol.search_ao_label(p))==3,'Expected three target p functions')
    selector=avas.AVAS(mf,labels,threshold=threshold,canonicalize=False)
    ncas,nelec,mo=selector.kernel();ncas,nelec=int(ncas),int(nelec)
    core=(mf.mol.nelectron-nelec)//2
    print(model['pdb_id'],label,f'CAS({nelec}e,{ncas}o)',flush=True)
    check(nelec%2==0 and 0<nelec<2*ncas,'Need occupied and virtual active orbitals')
    check(ncas<=CFG['max_active'],'AVAS active-orbital budget exceeded; no truncation applied')
    check(math.comb(ncas,nelec//2)**2<=CFG['max_determinants'],'CASCI determinant budget exceeded')
    overlap=mf.get_ovlp();active=mo[:,core:core+ncas]
    check(np.allclose(mo.T@overlap@mo,np.eye(mo.shape[1]),atol=1e-8),'Orbital orthonormality failed')
    population=float(np.trace(active.T@overlap@mf.make_rdm1()@overlap@active))
    check(abs(population-nelec)<1e-7,'Selected electron count inconsistent')
    molden.from_mo(mf.mol,str(folder/f'{label}_orbitals.molden'),mo)
    # Nonnegative Lowdin AO weights provide a reproducible orbital-character report.
    values,vectors=np.linalg.eigh(overlap)
    weights=((vectors*np.sqrt(values))@vectors.T@active)**2
    slices=mf.mol.aoslice_by_atom()[:,2:]
    capatoms=[i for i,m in enumerate(model['source_atom_mapping']) if m['kind']=='link_hydrogen']
    caprows=[j for i in capatoms for j in range(*slices[i])]
    targetrows=[j for i in model['target_atom_indices'] for j in range(*slices[i])]
    capweights=weights[caprows].sum(axis=0);targetweights=weights[targetrows].sum(axis=0)
    check(np.max(capweights)<0.25,'Selected orbital is dominated by a link cap')
    with (folder/f'{label}_orbital_character.csv').open('w') as stream:
        w=csv.writer(stream);w.writerow(['active_index','MO_column','target_atom_weight','link_cap_weight'])
        w.writerows((i,core+i,float(targetweights[i]),float(capweights[i])) for i in range(ncas))
    if CFG['make_cubes']:
        for i in range(ncas):cubegen.orbital(mf.mol,str(folder/f'{label}_{i}.cube'),active[:,i],nx=40,ny=40,nz=40)
    cas=mcscf.CASCI(exact,ncas,(nelec//2,nelec//2));cas.canonicalization=False
    cas.fcisolver.conv_tol=1e-10;cas.kernel(mo)
    check(cas.converged,'CASCI did not converge')
    h1,constant=cas.get_h1eff();h2=ao2mo.restore(1,cas.get_h2eff(),ncas)
    nocc=nelec//2
    determinant=float(constant+2*sum(h1[i,i] for i in range(nocc))+
        sum(2*h2[i,i,j,j]-h2[i,j,j,i] for i in range(nocc) for j in range(nocc)))
    check(abs(determinant-exact.energy_tot(dm=mf.make_rdm1()))<1e-7,'Frozen-core reconstruction failed')
    check(cas.e_tot<=determinant+1e-7,'Variational check failed')
    occupations=np.linalg.eigvalsh(cas.fcisolver.make_rdm1(cas.ci,ncas,cas.nelecas))
    check(abs(occupations.sum()-nelec)<1e-7,'Correlated electron count failed')
    report={'label':label,'threshold':threshold,'method':'AVAS O14/His NE2 2p; MINAO reference',
      'ncas':ncas,'nelecas':list(cas.nelecas),'ncore':core,'AO_labels':labels,
      'occupied_projector_weights':selector.occ_weights,'virtual_projector_weights':selector.vir_weights,
      'max_link_cap_weight':float(max(capweights)),'active_HF_population':population,
      'natural_occupations':occupations,'CASCI_Eh':float(cas.e_tot),'reference_determinant_Eh':determinant,
      'orbital_review':'Automatic target projection, orthonormality, electron count and cap-weight checks; Molden files available for visual inspection.'}
    save(folder/f'{label}_selection.json',report)
    return cas,report


## 9. Export and verify the qubit Hamiltonian

For each model and active space, write **FCIDUMP**, a NumPy integral archive, a Pauli-coefficient CSV and a validation JSON. Reload the actual FCIDUMP, solve its active Hamiltonian independently, and map it with Jordan-Wigner.

To avoid building an enormous full qubit matrix, the code applies the Pauli terms to the classical ground-state vector in the correct alpha/beta electron-number sector. It checks the energy, eigenstate residual and leakage outside that sector. This is a mapping verification, not VQE or a quantum-hardware run.

The scalar offset includes inactive electrons, nuclear repulsion and nuclear/environment electrostatics. It must be added **once** to a mapped electronic eigenvalue. The Pauli CSV does not include this offset.

In [10]:
from qiskit_nature.second_q.formats import fcidump_to_problem
from qiskit_nature.second_q.formats.fcidump import FCIDump
from qiskit_nature.second_q.mappers import JordanWignerMapper

def verify_mapped_state(operator, ci, n, particles, scalar, reference):
    """Apply Pauli terms in a fixed-particle sector without a full qubit matrix."""
    alpha=fci.cistring.make_strings(range(n),particles[0]).astype(np.uint64)
    beta=fci.cistring.make_strings(range(n),particles[1]).astype(np.uint64)
    states=(alpha[:,None] | (beta[None,:]<<n)).ravel()
    order=np.argsort(states);states=states[order]
    psi=np.asarray(ci).ravel()[order].astype(complex)
    check(abs(np.vdot(psi,psi).real-1)<1e-8,'CI state is not normalized')
    groups=defaultdict(list)
    for label,coefficient in operator.to_list():
        xmask=zmask=ny=0
        for bit,char in enumerate(reversed(label)):
            if char in 'XY':xmask|=1<<bit
            if char in 'ZY':zmask|=1<<bit
            if char=='Y':ny+=1
        groups[xmask].append((zmask,coefficient*(1j**ny)))
    action=np.zeros(len(states),complex);leakage2=0.0
    for xmask,terms in groups.items():
        factor=np.zeros(len(states),complex)
        for zmask,coefficient in terms:
            odd=np.bitwise_count(states & np.uint64(zmask)) & 1
            factor += coefficient*(1-2*odd.astype(np.int8))
        output=states ^ np.uint64(xmask)
        rows=np.searchsorted(states,output)
        valid=rows<len(states)
        valid[valid] &= states[rows[valid]]==output[valid]
        action[rows[valid]] += factor[valid]*psi[valid]
        # Each distinct flip mask produces distinct outputs per input; symmetry
        # cancellation occurs among terms within the same flip group.
        leakage2 += float(np.linalg.norm(factor[~valid]*psi[~valid])**2)
    expectation=float(np.vdot(psi,action).real+scalar)
    residual=float(np.linalg.norm(action-(reference-scalar)*psi))
    check(abs(expectation-reference)<1e-7,'Qubit expectation disagrees with CASCI')
    check(residual<2e-5,'Mapped eigenstate residual is too large')
    check(leakage2<1e-16,'Pauli terms leak outside the intended particle sector')
    return {'mapped_state_Eh':expectation,'mapped_residual_Eh':residual,
            'particle_sector_dimension':len(states),'outside_sector_squared_norm':leakage2}


def export_and_verify(cas,selection,folder,label):
    pending=folder/f'{label}.FCIDUMP.pending';final=folder/f'{label}.FCIDUMP'
    fcidump.from_mcscf(cas,str(pending),tol=1e-14,float_format=' %.17g')
    data=fcidump.read(str(pending),verbose=False)
    h1,constant=cas.get_h1eff();h2=ao2mo.restore(1,cas.get_h2eff(),cas.ncas)
    check(np.allclose(h1,data['H1'],atol=1e-12,rtol=0),'Exported h1 changed')
    check(np.allclose(h2,ao2mo.restore(1,data['H2'],cas.ncas),atol=1e-12,rtol=0),'Exported h2 changed')
    check(abs(constant-data['ECORE'])<1e-10,'Exported scalar changed')
    solver=fci.direct_spin1.FCI();solver.conv_tol=1e-10
    reloaded,ci=solver.kernel(data['H1'],data['H2'],cas.ncas,cas.nelecas,ecore=data['ECORE'])
    check(solver.converged and abs(reloaded-cas.e_tot)<1e-7,'Reloaded FCI disagrees with CASCI')
    problem=fcidump_to_problem(FCIDump.from_file(str(pending)))
    op=JordanWignerMapper().map(problem.hamiltonian.second_q_op()).simplify(atol=1e-12)
    offset=float(sum(problem.hamiltonian.constants.values()))
    check(tuple(problem.num_particles)==tuple(cas.nelecas),'Mapped electron count changed')
    check(abs(offset-constant)<1e-9,'Mapping lost the scalar offset')
    check(np.max(np.abs((op-op.adjoint()).simplify(atol=1e-10).coeffs))<1e-10,'Operator is not Hermitian')
    mapped=verify_mapped_state(op,ci,cas.ncas,cas.nelecas,offset,float(reloaded))
    with (folder/f'{label}_paulis.csv').open('w') as stream:
        w=csv.writer(stream);w.writerow(['pauli_label','real_Eh','imag_Eh'])
        w.writerows((p,float(c.real),float(c.imag)) for p,c in op.to_list())
    # Direct arrays are convenient if another framework is used for mapping.
    np.savez_compressed(folder/f'{label}_hamiltonian.npz',h1=h1,h2=h2,ecore=constant,
                        ncas=cas.ncas,nelec=np.array(cas.nelecas),mo_coeff=cas.mo_coeff,ncore=cas.ncore)
    pending.replace(final)
    report={**selection,**mapped,'reloaded_FCI_Eh':float(reloaded),'scalar_offset_Eh':offset,
        'qubits':op.num_qubits,'pauli_terms':len(op),'FCIDUMP_sha256':sha(final),
        'mapping_verified':True,'scalar_in_pauli_csv':False,
        'scalar_contents':'inactive electrons + QM nuclear repulsion + QM-nuclear/MM electrostatics',
        'excluded_energy_terms':'MM-MM, mechanical, restraint and van der Waals energies',
        'units':'hartree','h2_notation':'real spatial (pq|rs), chemist order',
        'mapping_mode_order':'Qiskit alpha block then beta block; Pauli rightmost character is qubit 0'}
    save(folder/f'{label}_validation.json',report)
    return report


def software_test():
    d=OUT/'software_test';d.mkdir(exist_ok=True)
    mol=gto.M(atom='Li 0 0 0; H 0 0 1.6',basis='sto-3g',verbose=0)
    mf=qmmm.mm_charge(scf.RHF(mol),[[4.,1.,0.]],[0.15],unit='Angstrom').run()
    check(mf.converged,'Software-test SCF failed')
    cas=mcscf.CASCI(mf,2,2);cas.fcisolver.conv_tol=1e-11;cas.run()
    report=export_and_verify(cas,{'purpose':'LiH software test, not Taxol','CASCI_Eh':float(cas.e_tot)},d,'LiH')
    check(abs(report['scalar_offset_Eh']-mf.energy_nuc())>1,'Frozen-core test is trivial')
    print('Embedded LiH export and qubit-state test passed.',flush=True)
    return report


## 10. Quick software check

Embedded LiH provides a quick test with frozen inactive electrons. It checks that export and mapping preserve the nontrivial constant. It is separate from the real Taxol results below.

In [11]:
SOFTWARE_TEST = software_test()

Embedded LiH export and qubit-state test passed.


## 11. Run the three real models

The next cell can take time on a laptop. Progress and SCF details are written into each structure's `scf.log`. Previously completed results are reused only when the exact model, basis, thresholds, versions and output hashes match. Restart the kernel after changing settings or code; set `force_recalculate=True` after editing the calculation implementation.

This is the production calculation path; no prepared-model placeholders remain for the default recipe.

In [12]:
def calculation_signature(model):
    # Includes the exact model, computational settings and implementation version.
    return fingerprint({'model':model,'basis':CFG['basis'],'thresholds':CFG['thresholds'],
        'implementation':'taxol-embedded-v2-2026-11-02','versions':VERSIONS})


def verified_cache(folder,model):
    path=folder/'complete.json'
    if not path.exists() or CFG['force_recalculate']:return None
    stamp=json.loads(path.read_text())
    if stamp.get('input_signature')!=calculation_signature(model):return None
    reports={}
    for name,digest in stamp.get('file_hashes',{}).items():
        p=folder/name
        if not p.exists() or sha(p)!=digest:return None
    for label in CFG['thresholds']:
        path=folder/f'{label}_validation.json'
        if not path.exists():return None
        r=json.loads(path.read_text())
        if not r.get('mapping_verified') or sha(folder/f'{label}.FCIDUMP')!=r['FCIDUMP_sha256']:return None
        # Read the actual portable export again, rather than trusting the filename.
        dump=FCIDump.from_file(str(folder/f'{label}.FCIDUMP'))
        problem=fcidump_to_problem(dump)
        check(problem.num_spatial_orbitals==r['ncas'],'Cached orbital count changed')
        check(tuple(problem.num_particles)==tuple(r['nelecas']),'Cached electron count changed')
        check(abs(float(sum(problem.hamiltonian.constants.values()))-r['scalar_offset_Eh'])<1e-9,
              'Cached scalar changed')
        reports[label]=r
    return reports


def stamp_completed(folder,model):
    patterns=['*.FCIDUMP','*_validation.json','*_selection.json','*_hamiltonian.npz',
              '*_orbitals.molden','*_orbital_character.csv','*_paulis.csv','mean_field.json']
    files=sorted({p for pattern in patterns for p in folder.glob(pattern)})
    save(folder/'complete.json',{'status':'completed','input_signature':calculation_signature(model),
         'model_sha256':sha(folder/'model.json'),'file_hashes':{p.name:sha(p) for p in files},
         'configuration':CFG,'versions':VERSIONS})


def run_calculations(models):
    results={}
    for pid,model in models.items():
        folder=OUT/pid
        cached=verified_cache(folder,model)
        if cached:
            print(pid,'reused validated results; all cached hashes and portable-file metadata checked.',flush=True)
            results[pid]=cached;continue
        print('Calculating',pid,'- progress is also written to scf.log',flush=True)
        mf,exact=run_scf(model,folder)
        results[pid]={}
        for label,threshold in CFG['thresholds'].items():
            cas,selection=select_and_solve(mf,exact,model,folder,label,threshold)
            results[pid][label]=export_and_verify(cas,selection,folder,label)
        stamp_completed(folder,model)
    return results


def summarize(results,models):
    records=[]
    for pid,spaces in results.items():
        for label,r in spaces.items():
            records.append({'structure':pid,'space':label,'active_electrons':sum(r['nelecas']),
                'spatial_orbitals':r['ncas'],'qubits':r['qubits'],'CASCI_Eh':r['CASCI_Eh'],
                'mapping_verified':r['mapping_verified'],'mapped_residual_Eh':r['mapped_residual_Eh'],
                'FCIDUMP':str(Path(pid)/f'{label}.FCIDUMP')})
        base,expanded=spaces['base'],spaces['expanded']
        save(OUT/pid/'sensitivity.json',{'base_orbitals':base['ncas'],'expanded_orbitals':expanded['ncas'],
            'expanded_minus_base_Eh':expanded['CASCI_Eh']-base['CASCI_Eh'],
            'interpretation':'Active-space sensitivity within this fixed model; not a binding or mutation free energy.',
            'expanded_is_larger':expanded['ncas']>base['ncas']})
    with (OUT/'summary.csv').open('w') as stream:
        writer=csv.DictWriter(stream,fieldnames=list(records[0]));writer.writeheader();writer.writerows(records)
    save(OUT/'run_manifest.json',{'status':'completed','configuration':CFG,'versions':VERSIONS,
        'input_file_sha256':{str(p.relative_to(INPUT)):sha(p) for p in sorted(INPUT.rglob('*.cif'))},
        'results':records,'warnings':{p:m['warnings'] for p,m in models.items()},
        'scientific_scope':'Static finite MMFF94-embedded contact models. No binding affinities or quantum advantage.'})
    print('Structure | space | electrons | orbitals | qubits | mapped energy verified')
    for r in records:
        print(' | '.join(str(r[k]) for k in ['structure','space','active_electrons','spatial_orbitals','qubits','mapping_verified']))
    return records


In [13]:
RESULTS = run_calculations(MODELS)
SUMMARY = summarize(RESULTS, MODELS)


9WDA reused validated results; all cached hashes and portable-file metadata checked.


9WDB reused validated results; all cached hashes and portable-file metadata checked.


1JFF reused validated results; all cached hashes and portable-file metadata checked.


Structure | space | electrons | orbitals | qubits | mapped energy verified
9WDA | base | 12 | 10 | 20 | True
9WDA | expanded | 12 | 11 | 22 | True
9WDB | base | 12 | 10 | 20 | True
9WDB | expanded | 12 | 11 | 22 | True
1JFF | base | 12 | 9 | 18 | True
1JFF | expanded | 12 | 11 | 22 | True


## 12. Send one file to the next quantum-computing stage

Use the real **9WDA/base.FCIDUMP** below to start. Its accompanying `base_validation.json` explains the particle count, scalar offset and classical reference. Use the matching 9WDB file for the mutant structure. Do not rebuild an unembedded Hamiltonian from the XYZ file, because that would discard this model's environment and orbital selection.

In [14]:
mapping_file = OUT / "9WDA" / "base.FCIDUMP"
problem = fcidump_to_problem(FCIDump.from_file(str(mapping_file)))
qubit_hamiltonian = JordanWignerMapper().map(problem.hamiltonian.second_q_op())
scalar_offset_Eh = float(sum(problem.hamiltonian.constants.values()))
print("Mapping file:", mapping_file)
print("Active particles (alpha, beta):", problem.num_particles)
print("Qubits:", qubit_hamiltonian.num_qubits)
print("Pauli terms:", len(qubit_hamiltonian))
print("Add this scalar once (hartree):", scalar_offset_Eh)
print("Classical reference (hartree):", RESULTS["9WDA"]["base"]["CASCI_Eh"])


Mapping file: /Users/srijan/Documents/ChatGPT/Quantum Computing/active_space_outputs/completed/9WDA/base.FCIDUMP
Active particles (alpha, beta): (6, 6)
Qubits: 20
Pauli terms: 14251
Add this scalar once (hartree): -668.7052943746193
Classical reference (hartree): -700.9668134295325


## What the results mean

You now have electronic Hamiltonians for explicitly defined Taxol-protein contact models, including a finite classical environment. The larger-space result measures sensitivity to active-space size; it does not prove convergence of basis, fragment size, protonation or embedding.

**Do not rank binding affinity from the total energies.** The structures have different geometries and environments. S239C sensitises tubulin to paclitaxel and has an allosteric mechanism; this static model does not establish that complete mechanism. The 1JFF contact-angle warning is preserved rather than describing it as a confirmed hydrogen bond.

Omitted: bulk protein outside the finite environment, waters/solvent, dynamics, MM-MM and mechanical energy terms in the electronic export, binding free-energy calculations, VQE optimisation and claims of quantum advantage. These are limitations of the defined proof of concept, not unfinished input forms.

Sources: [RCSB 9WDA](https://www.rcsb.org/structure/9WDA), [9WDB](https://www.rcsb.org/structure/9WDB), [1JFF](https://www.rcsb.org/structure/1JFF), [RDKit MMFF](https://www.rdkit.org/docs/source/rdkit.Chem.rdForceFieldHelpers.html), [PySCF embedding](https://pyscf.org/user/qmmm.html), [AVAS](https://pyscf.org/_modules/pyscf/mcscf/avas.html), [Qiskit FCIDUMP import](https://qiskit-community.github.io/qiskit-nature/stubs/qiskit_nature.second_q.formats.fcidump_to_problem.html).